# Booklet 2: Disambiguation with Stats

In this booklet, we will look at corpus-wide stats for disambiguation. The idea is simple: instead of staring at one sentence, we can run the model over a whole file and see (1) how much ambiguity the rules remove and (2) where the remaining ambiguity tends to come from. 


To illustrate the usefulness of this module, we will run it on a file containing ~1300 sentences from the Ojibwe People's Dictionary, located at `../data/parallel/oj_opd.txt`. Running on the entire file might take ~3 minutes, so if you want to test something quicker, there’s also a small sample at `../data/parallel/oj_50.txt`.


### Running the stats

We’ll call `disambiguate_with_stats()` on a *path* (this function reads the file as one block). Set `verbose=True` to print the stats tables. You’ll see a simple progress bar that reflects the current stage. 


In [1]:
# this block can be ignored, just setting up the path
import sys
import os
sys.path.append(os.path.abspath(".."))  

In [2]:
from src.stats_disambiguation import disambiguate_with_stats
from fst_runtime.fst import Fst

# FST and grammar paths (make sure these exist for your machine)
fst = Fst("../data/fst/ojibwe.att")  
grammar = "../data/rules/disambiguation.cg3"
# corpus file with ~1300 sentences
corpus_path = "../data/parallel/oj_opd.txt"

after_block, stats = disambiguate_with_stats(
    text_path=corpus_path,
    grammar=grammar,
    fst=fst,
    verbose=True, 
)

[############################] 4/4 All done!g text with cg3lmost all running time is spent here!)
|-------------------|----------|
| total words       | 5207     |
| readings before   | 8168     |
| readings after    | 6496     |
| analyses removed  | 1672     |
| ambiguity removed |    0.48  |
| ambiguous before  |    0.403 |
| ambiguous after   |    0.21  |

| type    |   words b |   words a |   readings b |   readings a |   removed |   avg b |   avg a |
|---------|-----------|-----------|--------------|--------------|-----------|---------|---------|
| verb    |      2447 |      2400 |         4528 |         3340 |      1188 |    1.85 |    1.39 |
| pronoun |       371 |       371 |          575 |          456 |       119 |    1.55 |    1.23 |
| noun    |      1163 |      1100 |         1631 |         1361 |       270 |    1.4  |    1.24 |
| adverb  |      1048 |      1042 |         1072 |         1042 |        30 |    1.02 |    1    |
| other   |       366 |       301 |          366 

### Output format

In the output, you will see the following tables:

1) Summary table: total words, readings (before/after), how many analyses were removed, and overall ambiguity percentages before vs after.
2) By-POS table: for each POS, average readings per token before and after.
3) Ambiguity overview table: split into four general buckets, lemma, preverb, POS, and morphological differences.
4) Top ambiguiities table: top ambiguous tokens and patterns section, this is most useful for improving the disambiguation and giving example tokens to debug on. 

### Analyzing the output

The above tables show that, by far, the most common ambiguity involves obviative number (e.g., `ObvSg` vs.`ObvPl`, `3PlObvObj` vs. `3SgObvObj`). This ambiguity is almost always inherent in surface forms, and we don’t want to force a choice in most contexts. So, this ambiguity ends up unresolved.

As for other ambiguities, a common high-ranking unresolved ambiguity is `0PlObj` vs. `0SgObj`. This is a number ambiguity for verbs encoding an inanimate object, and this is an ambiguity that should be solvable given the right context. After some manual checking, one would be able to see that the vast majority of unsolved cases stem from the fact that there is no overt inanimate noun in the context, so this ambiguity remains. 

Finally, expect the exact numbers to shift as both the FST and the CG3 rules evolve.

## Summary of the module

The `disambiguate_with_stats()` function gives you four detailed tables that summarize the performance of the disambiguation model on a certain text. Depending on your purpose, this module could be used to either get general performance statistics, or to get specific examples of unparsed ambiguities to push development. 